# 第20章：高级编程 - 高级编程技术 (Advanced Programming Techniques)

## Cambridge A-Level Computer Science (9618) - Paper 4

---

### 学习目标 (Learning Objectives)

通过本节课的学习，你将掌握：

1. **异常处理 (Exception Handling)** — try/except/finally, 自定义异常
2. **文件操作 (File Handling)** — 文本文件、二进制文件、CSV、JSON
3. **正则表达式 (Regular Expressions)** — 模式匹配与验证
4. **递归实践 (Recursion in Practice)** — 树遍历、目录遍历
5. **生成器与迭代器 (Generators and Iterators)**
6. **Lambda 函数与函数式编程 (Lambda and Functional Programming)**

---

### 为什么要学习这些高级技术？

这些技术是编写**健壮 (Robust)**、**高效 (Efficient)**、**可维护 (Maintainable)** 代码的关键。

- 异常处理让你的程序**不会因为意外错误而崩溃**
- 文件操作让你的程序**能够持久化存储数据**
- 正则表达式让你**高效地处理和验证文本**
- 递归让你**优雅地解决复杂的分治问题**

In [ ]:
# ============================================================
# 环境配置 - matplotlib 中文字体支持
# ============================================================
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import numpy as np
import os
import re
import json
import csv
import io
import tempfile

# 中文字体配置
matplotlib.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

print("✅ 环境配置完成！可以开始学习高级编程技术了。")

---

## 第一部分：异常处理 (Exception Handling)

### 1.1 什么是异常？

**异常 (Exception)** 是程序运行时发生的错误。如果不处理异常，程序会立即崩溃。

常见的异常类型：

| 异常类型 | 触发条件 | 示例 |
|---------|---------|------|
| `ValueError` | 值不合适 | `int("abc")` |
| `TypeError` | 类型错误 | `"hello" + 5` |
| `IndexError` | 索引越界 | `list[100]` |
| `KeyError` | 键不存在 | `dict["missing"]` |
| `FileNotFoundError` | 文件不存在 | `open("nofile.txt")` |
| `ZeroDivisionError` | 除以零 | `10 / 0` |
| `AttributeError` | 属性不存在 | `"hello".missing()` |

### 1.2 try/except/else/finally 语法

```python
try:
    # 可能出错的代码
except ExceptionType as e:
    # 处理特定异常
except Exception as e:
    # 处理所有其他异常
else:
    # 没有异常时执行
finally:
    # 无论是否有异常都执行（清理操作）
```

In [ ]:
# ============================================================
# 示例 1：异常处理基础
# ============================================================

print("=" * 60)
print("异常处理 (Exception Handling) 基础演示")
print("=" * 60)

# --- 1. 不处理异常时程序会崩溃 ---
print("\n--- 示例 1: 基本的 try/except ---")

def safe_divide(a, b):
    """
    安全的除法函数
    
    演示 try/except/else/finally 的完整用法
    """
    try:
        print(f"  尝试计算 {a} / {b}...")
        result = a / b
    except ZeroDivisionError:
        print(f"  ❌ 错误：不能除以零！")
        return None
    except TypeError as e:
        print(f"  ❌ 类型错误：{e}")
        return None
    else:
        # 只有在没有异常时才执行
        print(f"  ✅ 计算成功！")
        return result
    finally:
        # 无论如何都会执行
        print(f"  📋 [finally] 除法操作完成")

# 测试各种情况
print(f"结果: {safe_divide(10, 3)}")
print()
print(f"结果: {safe_divide(10, 0)}")
print()
print(f"结果: {safe_divide('hello', 2)}")

In [ ]:
# ============================================================
# 示例 2：自定义异常 (Custom Exceptions)
# ============================================================

print("=" * 60)
print("自定义异常 (Custom Exceptions) 演示")
print("=" * 60)


class ValidationError(Exception):
    """自定义验证错误"""
    def __init__(self, field, message):
        self.field = field
        self.message = message
        super().__init__(f"{field}: {message}")


class AgeError(ValidationError):
    """年龄验证错误"""
    pass


class EmailError(ValidationError):
    """邮箱验证错误"""
    pass


def register_user(name, age, email):
    """
    用户注册函数 - 使用自定义异常进行验证
    
    验证规则:
    - name 不能为空
    - age 必须在 0-150 之间
    - email 必须包含 '@'
    """
    if not name or not name.strip():
        raise ValidationError("name", "姓名不能为空")
    
    if not isinstance(age, int) or age < 0 or age > 150:
        raise AgeError("age", f"年龄 {age} 不合法（应在 0-150 之间）")
    
    if '@' not in email:
        raise EmailError("email", f"邮箱格式不正确: {email}")
    
    return {"name": name, "age": age, "email": email}


# --- 测试 ---
test_cases = [
    ("小明", 17, "xiaoming@email.com"),  # 正常
    ("", 20, "test@email.com"),            # 名字为空
    ("小红", -5, "xiaohong@email.com"),    # 年龄不合法
    ("小刚", 18, "xiaogang_email"),         # 邮箱格式错误
]

for name, age, email in test_cases:
    try:
        user = register_user(name, age, email)
        print(f"✅ 注册成功: {user}")
    except AgeError as e:
        print(f"❌ 年龄错误 - {e}")
    except EmailError as e:
        print(f"❌ 邮箱错误 - {e}")
    except ValidationError as e:
        print(f"❌ 验证错误 - {e}")

In [ ]:
# ============================================================
# 可视化：异常处理流程图
# ============================================================

fig, ax = plt.subplots(figsize=(14, 9))
ax.set_xlim(0, 14)
ax.set_ylim(0, 9)
ax.axis('off')
ax.set_title('异常处理流程 (Exception Handling Flow)', fontsize=16, fontweight='bold', pad=15)

# 流程框绘制函数
def draw_box(ax, x, y, w, h, text, color, text_color='white', fontsize=10):
    box = FancyBboxPatch((x - w/2, y - h/2), w, h, boxstyle='round,pad=0.15',
                          facecolor=color, edgecolor='#2c3e50', linewidth=2, alpha=0.9)
    ax.add_patch(box)
    ax.text(x, y, text, ha='center', va='center', fontsize=fontsize,
            color=text_color, fontweight='bold')

# 箭头
def draw_arrow(ax, x1, y1, x2, y2, label='', color='#2c3e50'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=2))
    if label:
        mx, my = (x1+x2)/2, (y1+y2)/2
        ax.text(mx + 0.2, my, label, fontsize=9, color=color, fontweight='bold')

# 流程图
draw_box(ax, 5, 8, 3, 0.8, 'try 代码块', '#3498db')
draw_arrow(ax, 5, 7.6, 5, 7.0)

# 判断菱形
diamond = plt.Polygon([[5, 6.8], [6.5, 6.2], [5, 5.6], [3.5, 6.2]], 
                       facecolor='#f39c12', edgecolor='#2c3e50', linewidth=2)
ax.add_patch(diamond)
ax.text(5, 6.2, '异常？', ha='center', va='center', fontsize=10, fontweight='bold', color='white')

# 有异常 -> except
draw_arrow(ax, 6.5, 6.2, 9, 6.2, '是 (Yes)', '#e74c3c')
draw_box(ax, 10.5, 6.2, 2.5, 0.8, 'except 处理异常', '#e74c3c')
draw_arrow(ax, 10.5, 5.8, 10.5, 4.5)

# 无异常 -> else
draw_arrow(ax, 5, 5.6, 5, 4.8, '否 (No)', '#2ecc71')
draw_box(ax, 5, 4.2, 3, 0.8, 'else 成功处理', '#2ecc71')
draw_arrow(ax, 5, 3.8, 5, 3.0)

# finally
draw_box(ax, 7, 2.2, 4, 0.8, 'finally 清理（始终执行）', '#9b59b6')
draw_arrow(ax, 5, 3.0, 7, 2.6)
draw_arrow(ax, 10.5, 4.5, 10.5, 2.6)
draw_arrow(ax, 10.5, 2.6, 9, 2.4)

# 继续
draw_arrow(ax, 7, 1.8, 7, 1.0)
draw_box(ax, 7, 0.5, 3, 0.7, '程序继续运行', '#34495e')

# 说明
ax.text(1, 4.5, '异常处理的四个部分:\n\n'
        '1. try: 可能出错的代码\n'
        '2. except: 处理异常\n'
        '3. else: 无异常时执行\n'
        '4. finally: 始终执行',
        fontsize=9, va='center', color='#2c3e50',
        bbox=dict(boxstyle='round', facecolor='#ecf0f1', edgecolor='#bdc3c7'))

plt.tight_layout()
plt.show()

---

## 第二部分：文件操作 (File Handling)

### 2.1 文件操作基础

Python 使用 `open()` 函数打开文件，支持多种模式：

| 模式 | 说明 | 文件不存在时 |
|------|------|------------|
| `'r'` | 只读 (Read) | 报错 |
| `'w'` | 写入 (Write) - 覆盖 | 创建新文件 |
| `'a'` | 追加 (Append) | 创建新文件 |
| `'x'` | 独占创建 (Exclusive) | 文件已存在则报错 |
| `'b'` | 二进制模式 | — |
| `'+'` | 读写模式 | — |

### 2.2 with 语句

**强烈建议**使用 `with` 语句操作文件，它会自动关闭文件：

```python
with open('file.txt', 'r') as f:
    content = f.read()
# 文件在这里已经自动关闭
```

In [ ]:
# ============================================================
# 示例 3：文本文件操作
# ============================================================

print("=" * 60)
print("文本文件操作 (Text File Handling) 演示")
print("=" * 60)

# 使用临时目录，避免影响用户文件
temp_dir = tempfile.mkdtemp()
print(f"临时目录: {temp_dir}")

# --- 写入文本文件 ---
filepath = os.path.join(temp_dir, "students.txt")

print("\n--- 写入文件 (Write) ---")
students_data = [
    "姓名,年龄,科目,成绩",
    "小明,17,Computer Science,92",
    "小红,16,Mathematics,88",
    "小刚,17,Physics,95",
    "小芳,16,Chemistry,90",
]

with open(filepath, 'w', encoding='utf-8') as f:
    for line in students_data:
        f.write(line + '\n')
print(f"✅ 数据已写入 {filepath}")

# --- 读取文本文件 ---
print("\n--- 读取文件 (Read) ---")

# 方法 1: read() - 读取整个文件
with open(filepath, 'r', encoding='utf-8') as f:
    content = f.read()
print("方法1 - read():")
print(content)

# 方法 2: readlines() - 读取所有行到列表
with open(filepath, 'r', encoding='utf-8') as f:
    lines = f.readlines()
print("方法2 - readlines():")
for i, line in enumerate(lines):
    print(f"  第{i}行: {line.strip()}")

# 方法 3: 逐行读取（推荐，内存友好）
print("\n方法3 - 逐行迭代:")
with open(filepath, 'r', encoding='utf-8') as f:
    for line_num, line in enumerate(f):
        print(f"  [{line_num}] {line.strip()}")

# --- 追加模式 ---
print("\n--- 追加文件 (Append) ---")
with open(filepath, 'a', encoding='utf-8') as f:
    f.write("小李,17,Biology,87\n")
print("✅ 新记录已追加")

# 验证追加
with open(filepath, 'r', encoding='utf-8') as f:
    print(f"文件现在有 {len(f.readlines())} 行")

In [ ]:
# ============================================================
# 示例 4：CSV 文件操作
# ============================================================

print("=" * 60)
print("CSV 文件操作演示")
print("=" * 60)

csv_path = os.path.join(temp_dir, "grades.csv")

# --- 写入 CSV ---
print("\n--- 写入 CSV ---")
headers = ['姓名', '数学', '英语', '计算机科学']
rows = [
    ['小明', 92, 88, 95],
    ['小红', 88, 95, 82],
    ['小刚', 95, 78, 90],
    ['小芳', 90, 92, 88],
]

with open(csv_path, 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(headers)   # 写入表头
    writer.writerows(rows)     # 写入所有数据行
print(f"✅ CSV 文件已写入: {csv_path}")

# --- 读取 CSV ---
print("\n--- 读取 CSV (csv.reader) ---")
with open(csv_path, 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    for row in reader:
        print(f"  {row}")

# --- 使用 DictReader ---
print("\n--- 读取 CSV (csv.DictReader) ---")
with open(csv_path, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        avg = (int(row['数学']) + int(row['英语']) + int(row['计算机科学'])) / 3
        print(f"  {row['姓名']}: 平均分 = {avg:.1f}")

In [ ]:
# ============================================================
# 示例 5：JSON 文件操作
# ============================================================

print("=" * 60)
print("JSON 文件操作演示")
print("=" * 60)

json_path = os.path.join(temp_dir, "school.json")

# --- 创建 JSON 数据 ---
school_data = {
    "学校名称": "Cambridge International School",
    "创建年份": 2020,
    "学生列表": [
        {
            "姓名": "小明",
            "年龄": 17,
            "科目": ["Computer Science", "Mathematics", "Physics"],
            "成绩": {"CS": 92, "Math": 88, "Physics": 95}
        },
        {
            "姓名": "小红",
            "年龄": 16,
            "科目": ["Mathematics", "Chemistry", "Biology"],
            "成绩": {"Math": 95, "Chem": 90, "Bio": 88}
        }
    ],
    "是否招生中": True
}

# --- 写入 JSON ---
print("\n--- 写入 JSON ---")
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(school_data, f, ensure_ascii=False, indent=2)
print(f"✅ JSON 文件已写入: {json_path}")

# --- 读取 JSON ---
print("\n--- 读取 JSON ---")
with open(json_path, 'r', encoding='utf-8') as f:
    loaded_data = json.load(f)

print(f"学校: {loaded_data['学校名称']}")
print(f"学生数量: {len(loaded_data['学生列表'])}")
for student in loaded_data['学生列表']:
    print(f"  {student['姓名']}:")
    print(f"    科目: {', '.join(student['科目'])}")
    print(f"    成绩: {student['成绩']}")

# --- JSON 字符串操作 ---
print("\n--- JSON 字符串 (json.dumps / json.loads) ---")
json_str = json.dumps(school_data, ensure_ascii=False, indent=2)
print("JSON 字符串 (前200个字符):")
print(json_str[:200] + "...")

# 从字符串解析
parsed = json.loads(json_str)
print(f"\n从字符串解析: 类型={type(parsed).__name__}, 键={list(parsed.keys())}")

In [ ]:
# ============================================================
# 可视化：文件类型对比
# ============================================================

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle('文件类型对比 (File Types Comparison)', fontsize=16, fontweight='bold', y=1.02)

file_types = [
    ('TXT 文本文件', '#3498db', '简单的纯文本\n\n优点: 简单、通用\n缺点: 无结构\n\n用途: 日志、配置',
     'name: 小明\nage: 17\nsubject: CS'),
    ('CSV 文件', '#e67e22', '逗号分隔值\n\n优点: 表格数据\n缺点: 无嵌套\n\n用途: 电子表格',
     '姓名,年龄\n小明,17\n小红,16'),
    ('JSON 文件', '#2ecc71', '结构化数据\n\n优点: 层次结构\n缺点: 较复杂\n\n用途: API、配置',
     '{"name": "小明",\n "age": 17,\n "subjects":[...]}'),
    ('Binary 二进制', '#9b59b6', '二进制数据\n\n优点: 紧凑高效\n缺点: 不可读\n\n用途: 图片、音频',
     '\\x89PNG\\r\\n\n\\x1a\\n...\n(字节数据)'),
]

for ax, (title, color, desc, example) in zip(axes, file_types):
    ax.set_xlim(0, 6)
    ax.set_ylim(0, 8)
    ax.axis('off')
    
    # 标题
    header = FancyBboxPatch((0.3, 6.8), 5.4, 0.9, boxstyle='round,pad=0.15',
                             facecolor=color, edgecolor=color, alpha=0.9)
    ax.add_patch(header)
    ax.text(3, 7.25, title, ha='center', va='center', fontsize=12, fontweight='bold', color='white')
    
    # 描述
    ax.text(3, 4.8, desc, ha='center', va='center', fontsize=9, linespacing=1.4)
    
    # 代码示例
    code_box = FancyBboxPatch((0.3, 0.3), 5.4, 2.2, boxstyle='round,pad=0.15',
                               facecolor='#2c3e50', edgecolor='#34495e', alpha=0.9)
    ax.add_patch(code_box)
    ax.text(3, 1.4, example, ha='center', va='center', fontsize=8, color='#2ecc71',
            fontfamily='monospace', linespacing=1.3)

plt.tight_layout()
plt.show()

---

## 第三部分：正则表达式 (Regular Expressions)

### 3.1 什么是正则表达式？

**正则表达式 (Regular Expression, Regex)** 是一种用于描述文本模式的特殊语法。它可以用来：

- **搜索**: 在文本中查找符合模式的内容
- **验证**: 检查文本是否符合特定格式（如邮箱、手机号）
- **替换**: 批量替换文本中的内容
- **提取**: 从文本中提取特定信息

### 3.2 基本语法

| 符号 | 含义 | 示例 |
|------|------|------|
| `.` | 任意单个字符 | `a.c` → "abc", "a1c" |
| `\d` | 数字 [0-9] | `\d{3}` → "123" |
| `\w` | 字母/数字/下划线 | `\w+` → "hello_123" |
| `\s` | 空白字符 | `\s+` → " " |
| `^` | 字符串开头 | `^Hello` |
| `$` | 字符串结尾 | `world$` |
| `*` | 0次或多次 | `ab*c` → "ac", "abc", "abbc" |
| `+` | 1次或多次 | `ab+c` → "abc", "abbc" |
| `?` | 0次或1次 | `colou?r` → "color", "colour" |
| `{n}` | 恰好n次 | `\d{4}` → "2024" |
| `{n,m}` | n到m次 | `\d{2,4}` → "12", "1234" |
| `[abc]` | 字符集 | `[aeiou]` → 元音字母 |
| `[^abc]` | 排除字符集 | `[^0-9]` → 非数字 |
| `(...)` | 捕获组 | `(\d+)-(\d+)` |
| `\|` | 或 | `cat\|dog` |

In [ ]:
# ============================================================
# 示例 6：正则表达式基础
# ============================================================

print("=" * 60)
print("正则表达式 (Regular Expressions) 基础演示")
print("=" * 60)

# --- 基本匹配 ---
print("\n--- 基本匹配函数 ---")

text = "我的手机号是 13812345678，邮箱是 student@cambridge.edu，生日是 2007-05-15"
print(f"原文: {text}")
print()

# re.search(): 搜索第一个匹配
phone = re.search(r'1[3-9]\d{9}', text)
if phone:
    print(f"re.search() 找到手机号: {phone.group()}")

# re.findall(): 找所有匹配
numbers = re.findall(r'\d+', text)
print(f"re.findall() 找到所有数字: {numbers}")

# re.match(): 从开头匹配
match = re.match(r'我的', text)
print(f"re.match() 从开头匹配 '我的': {match.group() if match else None}")

# re.sub(): 替换
masked = re.sub(r'1[3-9]\d{9}', '***手机号已隐藏***', text)
print(f"re.sub() 隐藏手机号: {masked}")

# re.split(): 分割
parts = re.split(r'[，,]', text)
print(f"re.split() 按逗号分割: {parts}")

In [ ]:
# ============================================================
# 示例 7：正则表达式 - 实用验证器 (Regex Tester)
# ============================================================

print("=" * 60)
print("正则表达式验证器 (Regex Validator) 演示")
print("=" * 60)


class RegexValidator:
    """
    正则表达式验证器 - 常用的文本验证工具
    
    包含邮箱、手机号、密码强度等常用验证规则
    """
    
    # 预编译的正则表达式模式
    PATTERNS = {
        'email': r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$',
        'phone_cn': r'^1[3-9]\d{9}$',
        'phone_uk': r'^\+?44[\d\s]{10,12}$',
        'date_iso': r'^\d{4}-(0[1-9]|1[0-2])-(0[1-9]|[12]\d|3[01])$',
        'ipv4': r'^(\d{1,3}\.){3}\d{1,3}$',
        'url': r'^https?://[\w.-]+\.[a-zA-Z]{2,}(/\S*)?$',
        'strong_password': r'^(?=.*[a-z])(?=.*[A-Z])(?=.*\d)(?=.*[@$!%*?&])[A-Za-z\d@$!%*?&]{8,}$',
    }
    
    @staticmethod
    def validate(pattern_name, text):
        """验证文本是否匹配指定的模式"""
        if pattern_name not in RegexValidator.PATTERNS:
            return f"未知模式: {pattern_name}"
        
        pattern = RegexValidator.PATTERNS[pattern_name]
        match = re.match(pattern, text)
        return match is not None
    
    @staticmethod
    def extract_info(text):
        """从文本中提取各种信息"""
        results = {}
        
        # 提取邮箱
        emails = re.findall(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', text)
        if emails:
            results['邮箱'] = emails
        
        # 提取手机号
        phones = re.findall(r'1[3-9]\d{9}', text)
        if phones:
            results['手机号'] = phones
        
        # 提取日期
        dates = re.findall(r'\d{4}[-/]\d{1,2}[-/]\d{1,2}', text)
        if dates:
            results['日期'] = dates
        
        # 提取 URL
        urls = re.findall(r'https?://[\w.-]+\.[a-zA-Z]{2,}(?:/\S*)?', text)
        if urls:
            results['URL'] = urls
        
        return results


# --- 验证测试 ---
print("\n--- 邮箱验证 ---")
emails = ['student@cambridge.edu', 'invalid-email', 'test@123.com', '@bad.com']
for email in emails:
    valid = RegexValidator.validate('email', email)
    status = '✅' if valid else '❌'
    print(f"  {status} {email}")

print("\n--- 密码强度验证 ---")
passwords = ['Abc12345!', 'weakpass', '12345678', 'NoSpecial1', 'Str0ng@Pass']
for pwd in passwords:
    valid = RegexValidator.validate('strong_password', pwd)
    status = '✅ 强密码' if valid else '❌ 弱密码'
    print(f"  {status}: {pwd}")

print("\n--- 信息提取 ---")
sample_text = """联系方式：邮箱 alice@example.com 或 bob@school.edu.cn，
手机 13912345678，网站 https://www.example.com/page，
会议日期 2024-03-15"""
extracted = RegexValidator.extract_info(sample_text)
for key, values in extracted.items():
    print(f"  {key}: {values}")

In [ ]:
# ============================================================
# 可视化：正则表达式常用模式速查
# ============================================================

fig, ax = plt.subplots(figsize=(14, 8))
ax.axis('off')
ax.set_title('正则表达式 (Regex) 常用模式速查表', fontsize=16, fontweight='bold', pad=15)

# 表格数据
table_data = [
    ['模式', '含义', '示例', '匹配结果'],
    ['.', '任意字符', 'a.c', 'abc, a1c, a-c'],
    ['\\d', '数字 [0-9]', '\\d{3}', '123, 456'],
    ['\\w', '字母/数字/下划线', '\\w+', 'hello_123'],
    ['\\s', '空白字符', 'a\\sb', 'a b, a\\tb'],
    ['^...$', '开头和结尾', '^\\d+$', '仅含数字'],
    ['[abc]', '字符集合', '[aeiou]', '元音字母'],
    ['[^abc]', '排除集合', '[^0-9]', '非数字字符'],
    ['*', '0次或多次', 'ab*c', 'ac, abc, abbc'],
    ['+', '1次或多次', 'ab+c', 'abc, abbc'],
    ['?', '0次或1次', 'colou?r', 'color, colour'],
    ['{n,m}', 'n到m次', '\\d{2,4}', '12, 123, 1234'],
    ['(...)', '捕获组', '(\\d+)-(\\d+)', '分组提取'],
]

colors = ['#3498db'] + ['#ecf0f1', 'white'] * 6
text_colors = ['white'] + ['#2c3e50'] * 12

table = ax.table(cellText=table_data, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.8)

for i, row in enumerate(table_data):
    for j in range(len(row)):
        cell = table[i, j]
        if i == 0:
            cell.set_facecolor('#3498db')
            cell.set_text_props(color='white', fontweight='bold')
        else:
            cell.set_facecolor('#ecf0f1' if i % 2 == 0 else 'white')
        cell.set_edgecolor('#bdc3c7')

plt.tight_layout()
plt.show()

---

## 第四部分：递归实践 (Recursion in Practice)

### 4.1 递归回顾

**递归 (Recursion)** 是函数调用自身的编程技术。每个递归函数需要：

1. **基本情况 (Base Case)**: 停止递归的条件
2. **递归情况 (Recursive Case)**: 将问题分解为更小的子问题

### 4.2 树遍历 (Tree Traversal)

树结构是递归最经典的应用场景。

In [ ]:
# ============================================================
# 示例 8：递归 - 树遍历
# ============================================================

print("=" * 60)
print("递归实践 - 树遍历 (Tree Traversal)")
print("=" * 60)


class TreeNode:
    """二叉树节点"""
    def __init__(self, value, left=None, right=None):
        self.value = value
        self.left = left
        self.right = right


def preorder(node, result=None):
    """前序遍历 (Pre-order): 根 → 左 → 右"""
    if result is None:
        result = []
    if node:
        result.append(node.value)     # 访问根
        preorder(node.left, result)    # 遍历左子树
        preorder(node.right, result)   # 遍历右子树
    return result


def inorder(node, result=None):
    """中序遍历 (In-order): 左 → 根 → 右"""
    if result is None:
        result = []
    if node:
        inorder(node.left, result)
        result.append(node.value)
        inorder(node.right, result)
    return result


def postorder(node, result=None):
    """后序遍历 (Post-order): 左 → 右 → 根"""
    if result is None:
        result = []
    if node:
        postorder(node.left, result)
        postorder(node.right, result)
        result.append(node.value)
    return result


# 构建一棵二叉树
#        1
#       / \
#      2   3
#     / \   \
#    4   5   6

root = TreeNode(1,
    TreeNode(2,
        TreeNode(4),
        TreeNode(5)),
    TreeNode(3,
        None,
        TreeNode(6)))

print(f"前序遍历 (Pre-order):  {preorder(root)}")
print(f"中序遍历 (In-order):   {inorder(root)}")
print(f"后序遍历 (Post-order): {postorder(root)}")

In [ ]:
# ============================================================
# 可视化：树结构和遍历顺序
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('二叉树遍历方式对比', fontsize=16, fontweight='bold', y=1.02)

# 树节点位置
positions = {
    1: (3, 5),
    2: (1.5, 3),
    3: (4.5, 3),
    4: (0.5, 1),
    5: (2.5, 1),
    6: (5.5, 1),
}

edges = [(1, 2), (1, 3), (2, 4), (2, 5), (3, 6)]

traversals = [
    ('前序 Pre-order\n(根→左→右)', [1, 2, 4, 5, 3, 6], '#e74c3c'),
    ('中序 In-order\n(左→根→右)', [4, 2, 5, 1, 3, 6], '#3498db'),
    ('后序 Post-order\n(左→右→根)', [4, 5, 2, 6, 3, 1], '#2ecc71'),
]

for ax, (title, order, color) in zip(axes, traversals):
    ax.set_xlim(-0.5, 6.5)
    ax.set_ylim(-0.5, 6.5)
    ax.axis('off')
    ax.set_title(title, fontsize=12, fontweight='bold')
    
    # 绘制边
    for parent, child in edges:
        px, py = positions[parent]
        cx, cy = positions[child]
        ax.plot([px, cx], [py, cy], color='#bdc3c7', lw=2, zorder=1)
    
    # 绘制节点（按遍历顺序标注）
    for val, (x, y) in positions.items():
        idx = order.index(val) + 1
        circle = plt.Circle((x, y), 0.45, color=color, alpha=0.85, zorder=2)
        ax.add_patch(circle)
        ax.text(x, y, str(val), ha='center', va='center', fontsize=14,
                fontweight='bold', color='white', zorder=3)
        # 遍历顺序标签
        ax.text(x + 0.5, y + 0.3, f'({idx})', ha='center', fontsize=9, color=color)
    
    # 底部显示顺序
    ax.text(3, -0.2, f"顺序: {' → '.join(map(str, order))}",
            ha='center', fontsize=10, color=color, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 示例 9：递归 - 目录遍历 (Directory Walking)
# ============================================================

print("=" * 60)
print("递归实践 - 目录遍历 (Directory Walking)")
print("=" * 60)


def create_demo_directory(base_path):
    """创建演示用的目录结构"""
    structure = {
        'project': {
            'src': {
                'main.py': '# 主程序',
                'utils.py': '# 工具函数',
                'models': {
                    'user.py': '# 用户模型',
                    'product.py': '# 产品模型',
                }
            },
            'tests': {
                'test_main.py': '# 测试主程序',
                'test_utils.py': '# 测试工具函数',
            },
            'README.md': '# 项目说明',
        }
    }
    
    def create_structure(path, structure):
        for name, content in structure.items():
            full_path = os.path.join(path, name)
            if isinstance(content, dict):
                os.makedirs(full_path, exist_ok=True)
                create_structure(full_path, content)
            else:
                with open(full_path, 'w') as f:
                    f.write(content)
    
    create_structure(base_path, structure)
    return os.path.join(base_path, 'project')


def walk_directory(path, indent=0):
    """
    递归遍历目录 - 打印树状结构
    
    这是递归的典型应用：
    - 基本情况: 当前是文件
    - 递归情况: 当前是目录，遍历其中的每个条目
    """
    prefix = '│   ' * indent
    name = os.path.basename(path)
    
    if os.path.isfile(path):
        print(f"{prefix}├── 📄 {name}")
    elif os.path.isdir(path):
        print(f"{prefix}📁 {name}/")
        entries = sorted(os.listdir(path))
        for entry in entries:
            walk_directory(os.path.join(path, entry), indent + 1)


# 创建并遍历演示目录
project_path = create_demo_directory(temp_dir)
print("\n目录结构:")
walk_directory(project_path)


# 统计函数 - 递归计算目录大小
def count_files_recursive(path):
    """递归统计目录中的文件数和总大小"""
    if os.path.isfile(path):
        return 1, os.path.getsize(path)
    
    total_files = 0
    total_size = 0
    for entry in os.listdir(path):
        files, size = count_files_recursive(os.path.join(path, entry))
        total_files += files
        total_size += size
    
    return total_files, total_size


files, size = count_files_recursive(project_path)
print(f"\n统计: {files} 个文件, 总大小 {size} 字节")

---

## 第五部分：生成器与迭代器 (Generators and Iterators)

### 5.1 迭代器 (Iterator)

**迭代器**是一个实现了 `__iter__()` 和 `__next__()` 方法的对象。它可以逐个返回数据，而不是一次性加载所有数据到内存。

### 5.2 生成器 (Generator)

**生成器**是一种特殊的迭代器，使用 `yield` 关键字。它是创建迭代器最简单的方式。

```python
def my_generator():
    yield 1
    yield 2
    yield 3
```

**关键优势**：生成器是**惰性求值 (Lazy Evaluation)** 的，只在需要时才计算下一个值，非常节省内存！

In [ ]:
# ============================================================
# 示例 10：生成器与迭代器
# ============================================================

print("=" * 60)
print("生成器与迭代器 (Generators & Iterators) 演示")
print("=" * 60)

# --- 5.1 基本生成器 ---
print("\n--- 基本生成器 ---")

def countdown(n):
    """倒计时生成器"""
    print(f"  [生成器启动] 从 {n} 开始倒计时")
    while n > 0:
        print(f"  [yield 前] 即将产出 {n}")
        yield n      # 暂停并返回值
        print(f"  [yield 后] 继续执行")
        n -= 1
    print("  [生成器结束] 发射！")

# 使用生成器
gen = countdown(3)
print(f"类型: {type(gen)}")
print(f"next(): {next(gen)}")
print(f"next(): {next(gen)}")
print(f"next(): {next(gen)}")

# --- 5.2 实用生成器：Fibonacci ---
print("\n--- Fibonacci 生成器 ---")

def fibonacci():
    """无限 Fibonacci 数列生成器"""
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

# 获取前 15 个 Fibonacci 数
fib = fibonacci()
fib_numbers = [next(fib) for _ in range(15)]
print(f"前15个 Fibonacci 数: {fib_numbers}")

# --- 5.3 生成器表达式 ---
print("\n--- 生成器表达式 vs 列表推导式 ---")

import sys

# 列表推导式 - 一次性创建所有元素
list_comp = [x ** 2 for x in range(1000)]
# 生成器表达式 - 惰性求值
gen_expr = (x ** 2 for x in range(1000))

print(f"列表推导式内存: {sys.getsizeof(list_comp)} 字节")
print(f"生成器表达式内存: {sys.getsizeof(gen_expr)} 字节")
print("生成器节省了大量内存！")

In [ ]:
# ============================================================
# 示例 11：生成器管道 (Generator Pipeline)
# ============================================================

print("=" * 60)
print("生成器管道 (Generator Pipeline) 演示")
print("=" * 60)


def read_data():
    """第一步：生成原始数据"""
    data = [
        "小明,92", "小红,88", "小刚,invalid", "小芳,95",
        "小李,78", "小王,error", "小赵,100", "小孙,65"
    ]
    for item in data:
        yield item


def parse_records(data_gen):
    """第二步：解析记录（过滤无效数据）"""
    for item in data_gen:
        parts = item.split(',')
        try:
            name = parts[0]
            score = int(parts[1])
            yield (name, score)
        except (ValueError, IndexError):
            print(f"  ⚠️ 跳过无效记录: {item}")


def filter_passing(records_gen, min_score=60):
    """第三步：过滤及格的学生"""
    for name, score in records_gen:
        if score >= min_score:
            yield (name, score)


def add_grade(records_gen):
    """第四步：添加等级"""
    for name, score in records_gen:
        if score >= 90:
            grade = 'A'
        elif score >= 80:
            grade = 'B'
        elif score >= 70:
            grade = 'C'
        else:
            grade = 'D'
        yield (name, score, grade)


# 构建管道: 读取 → 解析 → 过滤 → 评级
print("\n管道处理流程: 读取 → 解析 → 过滤及格 → 添加等级")
print()

pipeline = add_grade(filter_passing(parse_records(read_data())))

print("处理结果:")
for name, score, grade in pipeline:
    print(f"  {name}: {score}分 → 等级 {grade}")

---

## 第六部分：Lambda 函数与函数式编程 (Lambda & Functional Programming)

### 6.1 Lambda 函数

**Lambda 函数**是匿名的单行函数，语法为：

```python
lambda 参数: 表达式
```

### 6.2 函数式编程三大核心函数

| 函数 | 用途 | 示例 |
|------|------|------|
| `map()` | 对每个元素执行函数 | `map(lambda x: x*2, [1,2,3])` → [2,4,6] |
| `filter()` | 过滤满足条件的元素 | `filter(lambda x: x>0, [-1,2,-3,4])` → [2,4] |
| `sorted()` | 自定义排序 | `sorted(data, key=lambda x: x[1])` |

In [ ]:
# ============================================================
# 示例 12：Lambda 和函数式编程
# ============================================================

print("=" * 60)
print("Lambda 与函数式编程 (Functional Programming) 演示")
print("=" * 60)

# --- Lambda 基础 ---
print("\n--- Lambda 基础 ---")

# 普通函数
def double(x):
    return x * 2

# 等价的 lambda
double_lambda = lambda x: x * 2

print(f"普通函数: double(5) = {double(5)}")
print(f"Lambda:   double_lambda(5) = {double_lambda(5)}")

# 多参数 lambda
add = lambda x, y: x + y
print(f"多参数 lambda: add(3, 4) = {add(3, 4)}")

# --- map() ---
print("\n--- map() ---")
numbers = [1, 2, 3, 4, 5]
squared = list(map(lambda x: x ** 2, numbers))
print(f"原始: {numbers}")
print(f"平方: {squared}")

names = ['  Alice  ', ' Bob', 'Charlie  ']
cleaned = list(map(lambda s: s.strip().upper(), names))
print(f"清理后: {cleaned}")

# --- filter() ---
print("\n--- filter() ---")
scores = [45, 78, 92, 55, 88, 30, 100, 67]
passing = list(filter(lambda x: x >= 60, scores))
print(f"所有成绩: {scores}")
print(f"及格成绩: {passing}")

# --- sorted() with key ---
print("\n--- sorted() with lambda key ---")
students = [
    ('小明', 92), ('小红', 88), ('小刚', 95), ('小芳', 78)
]

# 按成绩排序
by_score = sorted(students, key=lambda s: s[1], reverse=True)
print("按成绩降序:")
for name, score in by_score:
    print(f"  {name}: {score}")

# --- reduce() ---
print("\n--- reduce() ---")
from functools import reduce

# 计算阶乘
factorial = reduce(lambda x, y: x * y, range(1, 6))
print(f"5! = {factorial}")

# 链式操作
print("\n--- 链式操作 (Chaining) ---")
data = range(1, 21)
# 找出1-20中偶数的平方和
result = reduce(
    lambda a, b: a + b,
    map(
        lambda x: x ** 2,
        filter(
            lambda x: x % 2 == 0,
            data
        )
    )
)
print(f"1-20中偶数的平方和: {result}")
print(f"验证: {sum(x**2 for x in range(1, 21) if x % 2 == 0)}")

In [ ]:
# ============================================================
# 可视化：函数式编程 map/filter/reduce 流程
# ============================================================

fig, axes = plt.subplots(3, 1, figsize=(14, 10))
fig.suptitle('函数式编程核心操作 (Functional Programming)', fontsize=16, fontweight='bold', y=1.02)

# --- map ---
ax = axes[0]
ax.set_xlim(0, 14)
ax.set_ylim(0, 3)
ax.axis('off')
ax.set_title('map(func, iterable) — 对每个元素应用函数', fontsize=12, fontweight='bold')

# 输入
for i, val in enumerate([1, 2, 3, 4, 5]):
    ax.add_patch(FancyBboxPatch((i*1.5 + 0.5, 2), 1, 0.7, boxstyle='round,pad=0.1',
                                 facecolor='#3498db', alpha=0.8))
    ax.text(i*1.5 + 1, 2.35, str(val), ha='center', va='center', fontsize=11, color='white', fontweight='bold')

# 函数
ax.add_patch(FancyBboxPatch((3.5, 1), 3, 0.7, boxstyle='round,pad=0.1',
                             facecolor='#f39c12', alpha=0.9))
ax.text(5, 1.35, 'x ** 2', ha='center', va='center', fontsize=11, fontfamily='monospace', fontweight='bold')

# 输出
for i, val in enumerate([1, 4, 9, 16, 25]):
    ax.add_patch(FancyBboxPatch((i*1.5 + 0.5, 0), 1, 0.7, boxstyle='round,pad=0.1',
                                 facecolor='#2ecc71', alpha=0.8))
    ax.text(i*1.5 + 1, 0.35, str(val), ha='center', va='center', fontsize=11, color='white', fontweight='bold')

ax.text(9, 2.35, '输入', fontsize=10, color='#3498db', fontweight='bold')
ax.text(9, 0.35, '输出', fontsize=10, color='#2ecc71', fontweight='bold')

# --- filter ---
ax = axes[1]
ax.set_xlim(0, 14)
ax.set_ylim(0, 3)
ax.axis('off')
ax.set_title('filter(func, iterable) — 过滤满足条件的元素', fontsize=12, fontweight='bold')

input_vals = [1, 2, 3, 4, 5, 6]
for i, val in enumerate(input_vals):
    color = '#3498db' if val % 2 == 0 else '#bdc3c7'
    ax.add_patch(FancyBboxPatch((i*1.3 + 0.5, 2), 1, 0.7, boxstyle='round,pad=0.1',
                                 facecolor=color, alpha=0.8))
    ax.text(i*1.3 + 1, 2.35, str(val), ha='center', va='center', fontsize=11, 
            color='white', fontweight='bold')

ax.add_patch(FancyBboxPatch((3.5, 1), 3, 0.7, boxstyle='round,pad=0.1',
                             facecolor='#f39c12', alpha=0.9))
ax.text(5, 1.35, 'x % 2 == 0', ha='center', va='center', fontsize=11, fontfamily='monospace', fontweight='bold')

for i, val in enumerate([2, 4, 6]):
    ax.add_patch(FancyBboxPatch((i*1.5 + 1.5, 0), 1, 0.7, boxstyle='round,pad=0.1',
                                 facecolor='#2ecc71', alpha=0.8))
    ax.text(i*1.5 + 2, 0.35, str(val), ha='center', va='center', fontsize=11, color='white', fontweight='bold')

# --- reduce ---
ax = axes[2]
ax.set_xlim(0, 14)
ax.set_ylim(0, 3)
ax.axis('off')
ax.set_title('reduce(func, iterable) — 累积运算', fontsize=12, fontweight='bold')

steps = ['1', '1+2=3', '3+3=6', '6+4=10', '10+5=15']
for i, step in enumerate(steps):
    color = '#e74c3c' if i == len(steps)-1 else '#3498db'
    ax.add_patch(FancyBboxPatch((i*2.5 + 0.2, 1), 2.2, 0.8, boxstyle='round,pad=0.1',
                                 facecolor=color, alpha=0.8))
    ax.text(i*2.5 + 1.3, 1.4, step, ha='center', va='center', fontsize=10,
            color='white', fontweight='bold', fontfamily='monospace')
    if i < len(steps) - 1:
        ax.annotate('', xy=(i*2.5 + 2.5, 1.4), xytext=(i*2.5 + 2.2, 1.4),
                    arrowprops=dict(arrowstyle='->', color='#7f8c8d', lw=2))

ax.text(12.5, 1.4, '= 15', fontsize=14, color='#e74c3c', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 综合可视化：高级编程技术知识地图
# ============================================================

fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 16)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('高级编程技术知识地图 (Knowledge Map)', fontsize=18, fontweight='bold', pad=20)

# 中心节点
center = plt.Circle((8, 5), 1.2, color='#2c3e50', alpha=0.9)
ax.add_patch(center)
ax.text(8, 5, '高级编程\n技术', ha='center', va='center', fontsize=14, 
        fontweight='bold', color='white')

# 六大分支
branches = [
    (3, 8.5, '异常处理', '#e74c3c', 'try/except/finally\n自定义异常\nraise'),
    (13, 8.5, '文件操作', '#3498db', 'TXT/CSV/JSON\n读写/追加\nwith 语句'),
    (1.5, 5, '正则表达式', '#2ecc71', '模式匹配\n搜索/替换\n验证'),
    (14.5, 5, '递归', '#f39c12', '基本情况\n递归情况\n树遍历'),
    (3, 1.5, '生成器', '#9b59b6', 'yield\n惰性求值\n管道'),
    (13, 1.5, 'Lambda', '#e67e22', 'map/filter\nreduce\n函数式编程'),
]

for x, y, title, color, desc in branches:
    # 连线
    ax.plot([8, x], [5, y], color=color, lw=2, alpha=0.5, zorder=1)
    
    # 分支节点
    box = FancyBboxPatch((x - 1.8, y - 0.9), 3.6, 1.8, boxstyle='round,pad=0.15',
                          facecolor=color, edgecolor='#2c3e50', linewidth=2, alpha=0.85, zorder=2)
    ax.add_patch(box)
    ax.text(x, y + 0.35, title, ha='center', va='center', fontsize=12, 
            fontweight='bold', color='white', zorder=3)
    ax.text(x, y - 0.35, desc, ha='center', va='center', fontsize=8, 
            color='white', zorder=3, linespacing=1.3)

plt.tight_layout()
plt.show()

---

## 练习题 (Practice Exercises)

### 练习 1：异常处理

编写一个 `safe_calculator()` 函数，能够安全地进行四则运算：
- 输入：两个数和一个运算符 (+, -, *, /)
- 使用 try/except 处理所有可能的错误
- 自定义一个 `CalculatorError` 异常类

### 练习 2：文件处理

创建一个学生成绩管理系统（使用 JSON 文件存储），支持：
- 添加学生
- 查询成绩
- 计算平均分
- 导出为 CSV

### 练习 3：正则表达式

编写以下验证函数：
- `validate_id_card(id)`: 验证中国身份证号（18位，最后一位可以是X）
- `validate_plate(plate)`: 验证中国车牌号
- `extract_hashtags(text)`: 从文本中提取所有 #话题#

In [ ]:
# ============================================================
# 练习 1：安全计算器
# ============================================================

class CalculatorError(Exception):
    """计算器自定义异常"""
    pass


def safe_calculator(a, b, operator):
    """
    安全计算器 - 请在这里实现
    
    参数:
        a: 第一个数
        b: 第二个数
        operator: 运算符 (+, -, *, /)
    
    要求:
    - 处理 ZeroDivisionError
    - 处理无效运算符
    - 处理非数字输入
    """
    # TODO: 请完成此函数
    pass


# 测试
# print(safe_calculator(10, 3, '+'))    # 13
# print(safe_calculator(10, 0, '/'))    # 错误
# print(safe_calculator('a', 3, '+'))   # 错误
# print(safe_calculator(10, 3, '%'))    # 错误

In [ ]:
# ============================================================
# 练习 2：学生成绩管理系统 (JSON)
# ============================================================

class GradeManager:
    """
    学生成绩管理系统 - 请在这里实现
    
    使用 JSON 文件存储数据
    """
    
    def __init__(self, filepath):
        # TODO: 初始化，加载已有数据或创建新文件
        pass
    
    def add_student(self, name, grades):
        # TODO: 添加学生和成绩
        pass
    
    def get_average(self, name):
        # TODO: 获取某个学生的平均分
        pass
    
    def export_csv(self, csv_path):
        # TODO: 导出为 CSV 文件
        pass


# 测试
# manager = GradeManager('grades.json')
# manager.add_student('小明', {'数学': 92, '英语': 88, 'CS': 95})
# print(manager.get_average('小明'))

In [ ]:
# ============================================================
# 练习 3：正则表达式验证
# ============================================================

def validate_id_card(id_number):
    """
    验证中国身份证号
    - 18位数字，最后一位可以是X/x
    - 前6位是地区码，接下来8位是出生日期
    """
    # TODO: 使用正则表达式验证
    pass


def extract_hashtags(text):
    """
    从文本中提取 #话题# 格式的标签
    """
    # TODO: 使用正则表达式提取
    pass


# 测试
# print(validate_id_card('11010119900307001X'))  # True
# print(validate_id_card('1234'))                # False
# print(extract_hashtags('今天学了#Python#和#正则表达式#，很有趣！'))

---

## A-Level 考试要点总结

### 异常处理 (Exception Handling)
- try/except/else/finally 的完整用法
- 何时以及为什么使用异常处理
- 自定义异常类的定义和使用

### 文件操作 (File Handling)
- 文本文件的读、写、追加
- CSV 和 JSON 文件格式
- `with` 语句的重要性（自动资源管理）

### 正则表达式 (Regex)
- 基本模式语法
- 常见的验证场景（邮箱、手机号、日期）
- re.search、re.findall、re.sub 的使用

### 递归 (Recursion)
- 基本情况与递归情况
- 树遍历（前序、中序、后序）
- 递归 vs 迭代的对比

### 函数式编程
- Lambda 表达式
- map、filter、reduce 函数
- 生成器的概念和 yield 关键字

---

**上一节**: [01_面向对象编程.ipynb](./01_面向对象编程.ipynb)

**下一节**: [03_软件工程实践.ipynb](./03_软件工程实践.ipynb) - 软件开发生命周期、测试、文档